# PS2 - Transfer Learning with MobileNetV2 (Feature Extraction + Fine-Tuning)
Dataset: TensorFlow Flower Photos (5 classes, downloaded automatically).

## 1. Setup and GPU / VRAM check

In [3]:
import pathlib
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers

IMG = (160, 160)
BATCH = 32
EPOCHS = 5
FINE_EPOCHS = 5
FINE_TUNE_LAYERS = 30
MIN_FREE_VRAM_MB = 2048
SEED = 42

keras.utils.set_random_seed(SEED)

## 2. Dataset

In [ ]:
path = keras.utils.get_file(
    "flower_photos.tgz",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
    extract=True,
)
data_dir = next(pathlib.Path(path).parent.rglob("daisy")).parent
print("Data directory:", data_dir)

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG, batch_size=BATCH, label_mode="int",
)
val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG, batch_size=BATCH, label_mode="int",
)
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 4))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Training samples")
plt.tight_layout()
plt.show()

## 3. Model - MobileNetV2 with a custom classification head

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=IMG + (3,), include_top=False, weights="imagenet"
)
base.trainable = False

inputs = keras.Input(shape=IMG + (3,))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters (stage 1): {trainable:,}")

## 4. Stage 1 - Feature extraction (base frozen)

In [ ]:
early_stop_1 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[early_stop_1], verbose=2,
)

## 5. Stage 2 - Fine-tuning (top 30 layers, very low learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base.layers[-FINE_TUNE_LAYERS:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Trainable parameters (stage 2): {trainable:,}")

early_stop_2 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=FINE_EPOCHS,
    callbacks=[early_stop_2], verbose=2,
)

## 6. Evaluation

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {acc:.4f}  Validation loss: {loss:.4f}")

hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
n1 = len(h1.history["loss"])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, metric in enumerate(["loss", "accuracy"]):
    ax[i].plot(hist[metric], label="train")
    ax[i].plot(hist["val_" + metric], label="validation")
    ax[i].axvline(n1 - 0.5, color="gray", linestyle="--", label="fine-tune start")
    ax[i].set_title(metric)
    ax[i].set_xlabel("epoch")
    ax[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
y_true = np.concatenate([labels.numpy() for _, labels in val_ds])
y_pred = model.predict(val_ds, verbose=0).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

plt.figure(figsize=(7, 6))
sns.heatmap(
    confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
for images, labels in val_ds.take(1):
    probs = model.predict(images, verbose=0)
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        p, t = int(probs[i].argmax()), int(labels[i].numpy())
        plt.title(
            f"P:{class_names[p]}\nA:{class_names[t]}",
            color="green" if p == t else "red", fontsize=9,
        )
        plt.axis("off")
plt.suptitle("Validation predictions (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

## 7. Save the model and single-image inference utility

In [ ]:
model.save("mobilenetv2_flowers.keras")
print("Saved model to mobilenetv2_flowers.keras")


def predict_image(image_path):
    img = keras.utils.load_img(image_path, target_size=IMG)
    arr = keras.utils.img_to_array(img)[None, ...]
    probs = model.predict(arr, verbose=0)[0]
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{class_names[probs.argmax()]} ({probs.max():.1%})")
    plt.show()
    return dict(zip(class_names, probs.round(4)))